# VeriCall — Execution-Verified Post-Training for Tool Calling
### Ground-Up Architecture with In-Context Candidate Schemas & Anti-Collapse DPO

This notebook implements the complete **VeriCall** post-training and evaluation pipeline on **Qwen2.5-7B-Instruct**:
1. **16 Enterprise REST Tool Schemas** (10 Training in Finance/Commerce/CRM, 6 Held-Out Unseen in Cloud Infra/Travel Logistics).
2. **In-Context Candidate Tool Schemas**: Every prompt injects 6 candidate JSON tool schemas (1 target + 5 domain decoys) under `Available Tools:`.
3. **ExecutionValidator**: 6-state error taxonomy (`passed`, `invalid_json`, `wrong_endpoint`, `missing_arg`, `hallucinated_arg`, `type_error`) with markdown fence & regex JSON recovery.
4. **Completion-Only SFT**: Masks prompt tokens so the model only trains on assistant completions (`{"name": "...", "arguments": {...}}`) with `save_strategy="no"`.
5. **Anti-Collapse DPO**: $\beta=0.1$, $\text{lr}=5\times 10^{-6}$, $\text{rpo\_alpha}=0.2$ NLL anchor via `RobustDPOTrainer` on 1,000 near-miss preference pairs.
6. **Pre-Flight Diagnostics**: 25-sample smoke checks after each stage.
7. **Multi-Model Evaluation**: Evaluates Base, Constrained Decoding, SFT, and SFT+DPO across $M=500$ held-out requests with 10,000 bootstrap CIs and paired McNemar tests.


### Stage 1: Environment Setup & Dependencies


In [ ]:
# Stage 1: Environment Setup & Dependencies
import os
# Force single-GPU visibility immediately before any CUDA/Torch initialization.
# Prevents HuggingFace Trainer from wrapping BitsAndBytes 4-bit models in torch.nn.DataParallel,
# which causes permanent multi-GPU deadlocks on multi-GPU instances (e.g. Kaggle GPU T4 x2).
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys
import types
import importlib

print("[*] Installing and verifying dependencies...")
!pip install -q "fastapi>=0.100.0" "pydantic>=2.0.0" "httpx>=0.24.0" "statsmodels>=0.14.0" "scipy>=1.10.0" "tabulate>=0.9.0"

# Clean up broken mocks & safely patch package availability for Python 3.13 / TRL
for pkg in ["llm_blender", "liger_kernel", "liger_kernel.transformers"]:
    if pkg in sys.modules:
        del sys.modules[pkg]

try:
    import transformers.utils.import_utils
    if not getattr(transformers.utils.import_utils, "_is_safe_patched", False):
        _orig_is_pkg = transformers.utils.import_utils._is_package_available
        def _safe_is_pkg(pkg_name, return_version=False):
            if pkg_name in ["liger_kernel", "llm_blender"]:
                return (False, "N/A") if return_version else False
            try:
                return _orig_is_pkg(pkg_name, return_version=return_version)
            except (ValueError, AttributeError):
                return (False, "N/A") if return_version else False
        transformers.utils.import_utils._is_package_available = _safe_is_pkg
        transformers.utils.import_utils._is_safe_patched = True
except Exception:
    pass

# Fix vision mapping in transformers
import transformers.models.auto.modeling_auto
if not hasattr(transformers.models.auto.modeling_auto, "MODEL_FOR_VISION_2_SEQ_MAPPING_NAMES"):
    transformers.models.auto.modeling_auto.MODEL_FOR_VISION_2_SEQ_MAPPING_NAMES = {}

# Neutralize TRL import check flags
try:
    import trl.import_utils
    trl.import_utils.is_llmblender_available = lambda: False
    trl.import_utils.is_liger_kernel_available = lambda: False
except Exception:
    pass

# Attempt unsloth fast install
try:
    import unsloth
    print("[✓] Unsloth is already installed.")
    USE_UNSLOTH = True
except ImportError:
    print("[*] Installing unsloth for 2x faster 4-bit LoRA training...")
    !pip install -q "transformers>=4.40.0" "peft>=0.9.0" "accelerate>=0.28.0" "bitsandbytes>=0.43.0"
    !pip install -q "trl<0.12.0" || pip install -q trl
    try:
        import unsloth
        USE_UNSLOTH = True
        print("[✓] Unsloth installed successfully.")
    except Exception as e:
        print(f"[!] Unsloth install fallback: using Hugging Face PEFT + TRL ({e})")
        USE_UNSLOTH = False

# Verify TRL trainers import cleanly without liger_kernel conflicts
try:
    from trl import SFTTrainer, DPOTrainer, SFTConfig, DPOConfig
    print("[✓] Verified TRL trainers imported cleanly without mock conflicts.")
except Exception as e:
    print(f"[!] Warning on TRL import: {e}")

# Verify and set working directory to repository root (Supports Kaggle, Colab, Local)
from pathlib import Path

if os.path.exists("/kaggle/working"):
    repo_dir = "/kaggle/working/vericall"
    if not os.path.exists(repo_dir):
        print(f"[*] Cloning repository to {repo_dir}...")
        os.system(f"git clone https://github.com/bhuvneshsahu01/vericall.git {repo_dir}")
    else:
        print(f"[*] Pulling latest changes into {repo_dir}...")
        os.system(f"cd {repo_dir} && git fetch origin && git reset --hard origin/master")
    os.chdir(repo_dir)
elif os.path.exists("/content"):
    repo_dir = "/content/vericall"
    if not os.path.exists(repo_dir):
        print(f"[*] Cloning repository to {repo_dir}...")
        os.system(f"git clone https://github.com/bhuvneshsahu01/vericall.git {repo_dir}")
    else:
        print(f"[*] Pulling latest changes into {repo_dir}...")
        os.system(f"cd {repo_dir} && git fetch origin && git reset --hard origin/master")
    os.chdir(repo_dir)
else:
    curr = Path.cwd()
    if (curr / "pyproject.toml").exists():
        print(f"[✓] Already at repository root: {curr}")
    elif (curr.parent / "pyproject.toml").exists():
        os.chdir(str(curr.parent))
    else:
        print(f"[!] Current directory: {curr}")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print(f"[✓] Environment ready. Working directory: {os.getcwd()}")
print(f"[✓] Dependency setup complete. Unsloth engine active: {USE_UNSLOTH}")

### Stage 2: In-Process FastAPI Mock Sandbox & ExecutionValidator (16 Endpoints)


In [ ]:
from vericall.sandbox.validator import ExecutionValidator
from vericall.sandbox.app import app, ENDPOINT_REGISTRY, get_tool_json_schema

validator = ExecutionValidator()
print(f"[✓] Initialized ExecutionValidator with {len(validator.registry)} endpoints across 5 domains.")

# Smoke test all 16 endpoints
test_endpoints = [
    ("issue_refund", {"transaction_id": "txn_12345678", "amount": 49.99, "currency": "USD", "reason": "CUSTOMER_RETURN", "metadata": {"rep": "rep_1"}}),
    ("create_invoice", {"client_id": "cli_1001", "issue_date": "2026-11-01", "due_date": "2026-11-15", "line_items": [{"description": "Dev", "unit_price": 100.0, "quantity": 1}], "currency": "USD"}),
    ("transfer_funds", {"source_account": "acc_100001", "destination_account": "acc_100002", "amount": 250.0, "transfer_method": "ACH", "memo": "Payment"}),
    ("audit_ledger_entries", {"ledger_id": "ledg_100001", "start_date": "2026-10-01", "end_date": "2026-10-31", "entry_type": "CREDIT", "filter_flags": {"reconciled_only": True}}),
    ("query_inventory", {"sku_list": ["SKU-PROD01"], "warehouse_id": "wh_us-01", "include_out_of_stock": False, "fulfillment_channel": "WAREHOUSE_DIRECT"}),
    ("apply_discount_code", {"cart_id": "cart_100001", "promo_code": "SAVE20", "channel": "ONLINE_WEB", "customer_loyalty_pin": 1234}),
    ("update_subscription", {"subscription_id": "sub_10000001", "plan_tier": "PRO_MONTHLY", "auto_renew": True, "addon_features": ["logs"], "effective_date": "2026-12-01"}),
    ("create_support_ticket", {"customer_id": "cust_10001", "priority": "HIGH", "department": "TECHNICAL", "subject": "Platform latency issue", "ticket_context": {"origin": "web"}}),
    ("update_customer_tier", {"customer_id": "cust_10001", "target_tier": "GOLD", "assigned_rep_id": "rep_1001", "override_reason": "High spending account", "benefits": ["waived_fees"]}),
    ("schedule_webhook", {"destination_url": "https://api.partner.com/events", "subscribed_events": ["order.created"], "retry_count": 3, "signing_secret": "whsec_12345678", "headers": {"X-App": "test"}}),
    ("provision_compute_node", {"instance_tier": "c6i.xlarge", "cloud_region": "us-east-1", "cluster_count": 2, "environment_tag": "PRODUCTION", "resource_tags": {"team": "infra"}}),
    ("scale_kubernetes_cluster", {"cluster_name": "k8s-prod-101", "node_pool": "worker-pool", "desired_replicas": 5, "scaling_strategy": "SURGE_FIRST", "autoscale_config": {"min_nodes": 2, "max_nodes": 20}}),
    ("rotate_iam_credentials", {"service_account_id": "sa-data1001", "credential_type": "OAUTH2_KEY", "grace_period_hours": 48, "notify_emails": ["ops@company.com"], "force_revocation": False}),
    ("search_multi_leg_flights", {"itinerary_segments": [{"departure_airport": "JFK", "arrival_airport": "LHR", "flight_date": "2026-11-10"}], "cabin_class": "BUSINESS", "passenger_count": 1, "flexible_dates": False, "preferred_alliances": ["STAR_ALLIANCE"]}),
    ("reserve_corporate_hotel", {"property_code": "HTL-NY-1001", "check_in_date": "2026-12-01", "check_out_date": "2026-12-05", "room_category": "EXECUTIVE_SUITE", "corporate_billing_code": "CORP-10001", "guest_manifest": [{"first_name": "Jane", "last_name": "Doe"}]}),
    ("cancel_itinerary_segment", {"booking_reference": "PNR-ABC123", "segment_id": "seg_01", "cancellation_reason": "CLIENT_RESCHEDULE", "request_credit_voucher": True, "refund_override_notes": "Authorized"}),
]

for fn, args in test_endpoints:
    res = validator.validate_call({"name": fn, "arguments": args}, expected_func=fn)
    assert res.is_success, f"Endpoint {fn} failed: {res.error_message}"

print("[✓] All 16 sandbox endpoints verified functional with HTTP 200 execution.")


### Stage 3: Load Model & Tokenizer (Dual Engine: Unsloth / PEFT 4-Bit)


In [ ]:
# Verify GPU is available
import torch
assert torch.cuda.is_available(), "ERROR: Kaggle GPU is not enabled! Turn on Accelerator -> GPU T4 x2 in Kaggle Settings."
if torch.cuda.is_available():
    torch.cuda.set_device(0)
print(f"[✓] GPU available: {torch.cuda.get_device_name(0)} (Count: {torch.cuda.device_count()})")
import torch
from transformers import AutoTokenizer

model_id = "Qwen/Qwen2.5-7B-Instruct"
max_seq_length = 1024

print(f"[*] Loading {model_id} in 4-bit...")

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_id,
        max_seq_length=max_seq_length,
        load_in_4bit=True,
    )
    print("[✓] Loaded model via unsloth FastLanguageModel.")
else:
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    # Target GPU 0 to prevent cross-device pipeline deadlock with gradient checkpointing
    target_device = {"": 0} if torch.cuda.is_available() else "auto"
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map=target_device,
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    print("[✓] Loaded model via standard HuggingFace BitsAndBytes.")

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"[✓] Tokenizer configured (vocab_size={len(tokenizer)}). Ready for evaluation & training.")

### Stage 4: Base Model & Constrained Decoding Evaluation (with 25-Sample Pre-Flight Gate)


In [ ]:
# Stage 4: Base Model & Constrained Decoding Evaluation (with Fast Batched Generation & JSON Stopping)
import json
import os
import gc
import re
from pathlib import Path
from tqdm import tqdm
import torch
from transformers import StoppingCriteria, StoppingCriteriaList
from vericall.eval.evaluate import evaluate_predictions, apply_constrained_decoding

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if hasattr(model, 'config'):
    model.config.use_cache = True
if hasattr(model, 'eval'):
    model.eval()

# Custom JSON stopping criteria: stops generation once the outer JSON object closes
class JsonBraceStoppingCriteria(StoppingCriteria):
    def __init__(self, tokenizer, prompt_len: int):
        self.tokenizer = tokenizer
        self.prompt_len = prompt_len
        self.im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        last_tok = input_ids[0, -1].item()
        if self.im_end_id is not None and last_tok == self.im_end_id:
            return True
        gen_tokens = input_ids[0, self.prompt_len:].tolist()
        if len(gen_tokens) > 15:
            text = self.tokenizer.decode(gen_tokens, skip_special_tokens=False)
            if "{" in text and "}" in text:
                depth = 0
                started = False
                for ch in text:
                    if ch == "{":
                        depth += 1
                        started = True
                    elif ch == "}":
                        depth -= 1
                        if started and depth == 0:
                            return True
        return False

def generate_batched(model, tokenizer, prompts, batch_size=8, max_new_tokens=140, desc="Generating"):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    orig_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    eos_ids = [tokenizer.eos_token_id] + ([im_end_id] if im_end_id is not None and im_end_id != tokenizer.eos_token_id else [])

    dev = getattr(model, "device", next(model.parameters()).device)
    responses = []

    curr_batch_size = batch_size
    i = 0
    pbar = tqdm(total=len(prompts), desc=desc) if len(prompts) > curr_batch_size else None

    while i < len(prompts):
        batch_prompts = prompts[i:i + curr_batch_size]
        try:
            inputs = tokenizer(
                batch_prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=1024,
            ).to(dev)

            with torch.inference_mode(), torch.amp.autocast("cuda", dtype=torch.float16 if torch.cuda.is_available() else torch.float32):
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    use_cache=True,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=eos_ids,
                )

            input_len = inputs.input_ids.shape[1]
            for b in range(len(batch_prompts)):
                gen_tokens = outputs[b][input_len:]
                text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
                responses.append(text)

            if pbar:
                pbar.update(len(batch_prompts))
            i += len(batch_prompts)

        except torch.cuda.OutOfMemoryError:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            if curr_batch_size > 1:
                curr_batch_size = max(1, curr_batch_size // 2)
                print(f"\n[!] CUDA OOM detected. Halving batch_size to {curr_batch_size} and retrying...")
            else:
                raise

    if pbar:
        pbar.close()

    tokenizer.padding_side = orig_padding_side
    return responses

# 1. Load M=500 unseen API test set (each item has prompt with 6 candidate schemas)
unseen_path = Path("data/eval_unseen_apis.jsonl")
test_records = [json.loads(line) for line in open(unseen_path, "r", encoding="utf-8") if line.strip()]
print(f"[*] Loaded {len(test_records)} frozen unseen API evaluation requests.")
assert "Available Tools:" in test_records[0]["prompt"], "Prompt missing candidate tool schemas!"
print(f"[✓] Verified in-prompt candidate schemas present. Target: {test_records[0]['target_function']}")

# 2. Load BFCL AST evaluation requests (N=200)
bfcl_path = Path("data/eval_bfcl_ast_subset.jsonl")
if not bfcl_path.exists():
    bfcl_path = Path("data/eval_bfcl_subset.jsonl")
bfcl_records = [json.loads(line) for line in open(bfcl_path, "r", encoding="utf-8") if line.strip()]
print(f"[*] Loaded {len(bfcl_records)} frozen BFCL AST evaluation requests.")

bfcl_prompts = [item["prompt"] for item in bfcl_records]

# --- 25-SAMPLE PRE-FLIGHT GATE ---
print("\n[*] Running 25-sample Base Model Pre-Flight Smoke Check (batched)...")
smoke_prompts = [item["prompt"] for item in test_records[:25]]
smoke_preds = generate_batched(model, tokenizer, smoke_prompts, batch_size=8, max_new_tokens=140, desc="Base Smoke (25)")
smoke_res = evaluate_predictions(test_records[:25], smoke_preds, "base_smoke")
base_smoke_acc = smoke_res["accuracy_pct"]
print(f"[Base Smoke Check] Passed: {smoke_res['passed_requests']}/25 ({base_smoke_acc}%)")
print(f"                   Errors: {smoke_res['error_taxonomy']}")

# --- FULL 500-SAMPLE EVALUATION (BATCHED) ---
print("\n[*] Generating predictions on full M=500 unseen API test set with Base Model (batched, 8x faster)...")
rest_prompts = [item["prompt"] for item in test_records[25:]]
rest_preds = generate_batched(model, tokenizer, rest_prompts, batch_size=8, max_new_tokens=140, desc="Base Full Eval (475)")
base_predictions = smoke_preds + rest_preds

# Evaluate Base on BFCL AST Subset (N=200, realistic 6 candidate tools per prompt)
print("\n[*] Evaluating Base Model on BFCL AST subset (N=200, batched)...")
bfcl_prompts = [item["prompt"] for item in bfcl_records]
bfcl_base_preds = generate_batched(model, tokenizer, bfcl_prompts, batch_size=8, max_new_tokens=140, desc="Base BFCL Eval (200)")

# Evaluate Base Unseen
base_results = evaluate_predictions(test_records, base_predictions, "base_model")
print(f"\n[Base Model] Unseen Accuracy: {base_results['accuracy_pct']}% (Passed: {base_results['passed_requests']}/{base_results['total_requests']})")
print(f"             95% Bootstrap CI: [{base_results['bootstrap_95_ci'][0]}%, {base_results['bootstrap_95_ci'][1]}%]")
print(f"             Errors: {base_results['error_taxonomy']}")

# Evaluate Base + Constrained Decoding (strips markdown fences, repairs truncated braces, coerces types)
print("\n[*] Evaluating Base Model + Constrained Schema Decoding...")
constrained_preds = []
for item, pred in zip(test_records, base_predictions):
    cand_tools = item["candidate_tools"]
    cleaned = apply_constrained_decoding(pred, cand_tools, user_query=item.get("query"))
    constrained_preds.append(cleaned)

constrained_results = evaluate_predictions(test_records, constrained_preds, "base_constrained")
print(f"[Constrained Decoding] Accuracy: {constrained_results['accuracy_pct']}% (Passed: {constrained_results['passed_requests']}/{constrained_results['total_requests']})")
print(f"                      95% CI: [{constrained_results['bootstrap_95_ci'][0]}%, {constrained_results['bootstrap_95_ci'][1]}%]")
print(f"                      Errors: {constrained_results['error_taxonomy']}")

# Evaluate Base BFCL
bfcl_base_results = evaluate_predictions(bfcl_records, bfcl_base_preds, "bfcl_base")
bfcl_base_acc_pct = bfcl_base_results["accuracy_pct"]
print(f"[Base Model BFCL] Accuracy: {bfcl_base_acc_pct}% (Passed: {bfcl_base_results['passed_requests']}/{bfcl_base_results['total_requests']})")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


### Stage 5: QLoRA Supervised Fine-Tuning (SFT) with Completion-Only Loss Masking


In [ ]:
# Stage 5: Native Completion-Sliced QLoRA Supervised Fine-Tuning (SFT)
import os
import sys
import json
import math
import gc
from pathlib import Path
from tqdm import tqdm
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

# Helper function: completion-sliced lm_head logps (1.83 GB -> 18 MB VRAM)
def compute_completion_logps(model, input_ids: torch.Tensor, attention_mask: torch.Tensor, prompt_len: int):
    """
    Runs the transformer backbone on full sequence, then slices ONLY the ~60 completion
    tokens before projecting through lm_head. Slashes peak tensor from 1.83 GiB to 18 MB!
    """
    causal_lm = model.get_base_model() if hasattr(model, "get_base_model") else model
    if hasattr(causal_lm, "model") and hasattr(causal_lm, "lm_head"):
        backbone = causal_lm.model
        head = causal_lm.lm_head
    elif hasattr(model, "model") and hasattr(model, "lm_head"):
        backbone = model.model
        head = model.lm_head
    else:
        raise AttributeError("Cannot locate backbone model and lm_head")

    out = backbone(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
    hidden = out[0]
    comp_hidden = hidden[:, prompt_len - 1 : -1, :]
    comp_labels = input_ids[:, prompt_len:]
    comp_logits = head(comp_hidden)
    per_token_logps = -torch.nn.functional.cross_entropy(
        comp_logits.reshape(-1, comp_logits.size(-1)).float(),
        comp_labels.reshape(-1),
        reduction="none",
    )
    return per_token_logps.sum(), per_token_logps.mean()

# 1. Initialize LoRA targeting linear projections if not already initialized
if not isinstance(model, PeftModel):
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False})
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, peft_config)

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[✓] Initialized LoRA adapter. Trainable parameters: {trainable_params:,}")
assert trainable_params > 0, "FATAL: LoRA initialization failed, model has ZERO trainable parameters!"

# 2. Load N=2,000 SFT Training Records & Pre-tokenize
sft_train_path = Path("data/train_sft.jsonl")
train_raw = [json.loads(line) for line in open(sft_train_path, "r", encoding="utf-8") if line.strip()]
print(f"[*] Loaded {len(train_raw)} training samples from {sft_train_path.name}")

dev = getattr(model, "device", next(model.parameters()).device)

sft_samples = []
for item in train_raw:
    p_text = item["prompt"]
    c_text = json.dumps({"name": item["target_function"], "arguments": item["target_arguments"]}) + "<|im_end|>"
    p_ids = tokenizer.encode(p_text, add_special_tokens=False)
    c_ids = tokenizer.encode(c_text, add_special_tokens=False)
    full_ids = p_ids + c_ids
    sft_samples.append({
        "input_ids": torch.tensor([full_ids], dtype=torch.long, device=dev),
        "attention_mask": torch.ones((1, len(full_ids)), dtype=torch.long, device=dev),
        "prompt_len": len(p_ids),
    })

# 3. Configure Native Training Loop
num_epochs = 1
grad_accum = 8
total_steps = len(sft_samples) // grad_accum
learning_rate = 2e-4
warmup_steps = 10

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=learning_rate, weight_decay=0.01)

def get_lr(step):
    if step < warmup_steps:
        return learning_rate * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return learning_rate * 0.5 * (1.0 + math.cos(math.pi * progress))

scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

model.train()
if hasattr(model, 'config'):
    model.config.use_cache = False

print("[*] Starting Completion-Sliced QLoRA SFT training (10x faster, ~18 MB logits VRAM)...")
pbar = tqdm(total=total_steps, desc="SFT Progress")
running_loss = 0.0
optimizer.zero_grad()

for i, sample in enumerate(sft_samples):
    with torch.amp.autocast("cuda", dtype=torch.float16 if torch.cuda.is_available() else torch.float32):
        _, mean_lp = compute_completion_logps(
            model,
            sample["input_ids"],
            sample["attention_mask"],
            sample["prompt_len"],
        )
        loss = (-mean_lp) / grad_accum

    scaler.scale(loss).backward()
    running_loss += loss.item() * grad_accum

    if (i + 1) % grad_accum == 0 or (i + 1) == len(sft_samples):
        current_step = (i + 1) // grad_accum
        current_lr = get_lr(current_step)
        for param_group in optimizer.param_groups:
            param_group["lr"] = current_lr

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        avg_loss = running_loss / grad_accum
        pbar.set_postfix({"loss": f"{avg_loss:.4f}", "lr": f"{current_lr:.2e}"})
        pbar.update(1)
        running_loss = 0.0

pbar.close()

model.save_pretrained("checkpoints/sft_adapter")
tokenizer.save_pretrained("checkpoints/sft_adapter")
print("[✓] SFT training complete. Adapters saved to checkpoints/sft_adapter.")
del sft_samples
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


### Stage 6: Evaluate SFT Model on M=500 Unseen APIs (with Pre-Flight Gate)


In [ ]:
# Stage 6: Evaluate SFT Model on M=500 Unseen APIs & BFCL AST Subset
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
if hasattr(model, 'config'):
    model.config.use_cache = True
if hasattr(model, 'eval'):
    model.eval()

device = next(model.parameters()).device

tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# --- 25-SAMPLE PRE-FLIGHT GATE ---
print("\n[*] Running 25-sample SFT Pre-Flight Smoke Check (batched)...")
smoke_sft_prompts = [item["prompt"] for item in test_records[:25]]
smoke_sft_preds = generate_batched(model, tokenizer, smoke_sft_prompts, batch_size=8, max_new_tokens=140, desc="SFT Smoke (25)")

smoke_sft_res = evaluate_predictions(test_records[:25], smoke_sft_preds, "sft_smoke")
sft_smoke_acc = smoke_sft_res["accuracy_pct"]
print(f"[SFT Smoke Check] Passed: {smoke_sft_res['passed_requests']}/25 ({sft_smoke_acc}%) vs Base ({base_smoke_acc}%)")
print(f"                  Errors: {smoke_sft_res['error_taxonomy']}")

if sft_smoke_acc < base_smoke_acc:
    print(f"[!] Note: SFT smoke check on 25 samples is {sft_smoke_acc}% (Base: {base_smoke_acc}%). Proceeding to full M=500 evaluation.")
else:
    print(f"[✓] PRE-FLIGHT GATE PASSED: SFT accuracy ({sft_smoke_acc}%) matches/exceeds Base ({base_smoke_acc}%).")

# --- FULL 500-SAMPLE EVALUATION (BATCHED) ---
print("\n[*] Generating predictions on full M=500 unseen API test set with SFT Model (batched, 8x faster)...")
rest_sft_prompts = [item["prompt"] for item in test_records[25:]]
rest_sft_preds = generate_batched(model, tokenizer, rest_sft_prompts, batch_size=8, max_new_tokens=140, desc="SFT Full Eval (475)")
sft_predictions = smoke_sft_preds + rest_sft_preds

sft_results = evaluate_predictions(test_records, sft_predictions, "sft_only")
print(f"\n[SFT Model] Unseen Accuracy: {sft_results['accuracy_pct']}% (Passed: {sft_results['passed_requests']}/{sft_results['total_requests']})")
print(f"            95% Bootstrap CI: [{sft_results['bootstrap_95_ci'][0]}%, {sft_results['bootstrap_95_ci'][1]}%]")
print(f"            Errors: {sft_results['error_taxonomy']}")

# --- BFCL AST SUBSET EVALUATION (SFT) ---
print("\n[*] Evaluating SFT Model on BFCL AST subset (N=200, batched)...")
bfcl_sft_preds = generate_batched(model, tokenizer, bfcl_prompts, batch_size=8, max_new_tokens=140, desc="SFT BFCL Eval (200)")
bfcl_sft_results = evaluate_predictions(bfcl_records, bfcl_sft_preds, "bfcl_sft")
bfcl_sft_acc_pct = bfcl_sft_results["accuracy_pct"]
print(f"[SFT Model BFCL] Accuracy: {bfcl_sft_acc_pct}% (Passed: {bfcl_sft_results['passed_requests']}/{bfcl_sft_results['total_requests']})")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


### Stage 7: Execution-Grounded Near-Miss DPO Mining


In [ ]:
from vericall.data.mine_dpo import mine_dpo_pairs

dpo_pairs = mine_dpo_pairs(num_pairs=1000)
print(f"[✓] Successfully loaded/mined {len(dpo_pairs)} execution-verified preference pairs.")
print("Sample Preference Pair:")
print("  Prompt (first 120 chars):", dpo_pairs[0]["prompt"][:120].replace('\n', ' '))
print("  Chosen:                  ", dpo_pairs[0]["chosen"])
print("  Rejected:                ", dpo_pairs[0]["rejected"], f"({dpo_pairs[0]['rejected_status']})")


### Stage 8: QLoRA Direct Preference Optimization (DPO) with Anti-Collapse Safeguards


In [ ]:
# Stage 8: Native Completion-Sliced QLoRA Direct Preference Optimization (DPO) with RPO Anchor
import os
import sys
import json
import math
import gc
from pathlib import Path
from tqdm import tqdm
import torch

dev = getattr(model, "device", next(model.parameters()).device)

# 1. Load 1,000 DPO Pairs
dpo_file = Path("data/train_dpo.jsonl")
dpo_records = [json.loads(line) for line in open(dpo_file, "r", encoding="utf-8") if line.strip()]
print(f"[*] Loaded {len(dpo_records)} execution-grounded DPO pairs.")

dpo_tokenized = []
for r in dpo_records:
    p_text = r["prompt"]
    c_text = r["chosen"] if r["chosen"].endswith("<|im_end|>") else r["chosen"] + "<|im_end|>"
    rej_text = r["rejected"] if r["rejected"].endswith("<|im_end|>") else r["rejected"] + "<|im_end|>"

    p_ids = tokenizer.encode(p_text, add_special_tokens=False)
    c_ids = tokenizer.encode(c_text, add_special_tokens=False)
    rej_ids = tokenizer.encode(rej_text, add_special_tokens=False)

    full_c = p_ids + c_ids
    full_rej = p_ids + rej_ids

    dpo_tokenized.append({
        "c_ids": torch.tensor([full_c], dtype=torch.long, device=dev),
        "c_mask": torch.ones((1, len(full_c)), dtype=torch.long, device=dev),
        "rej_ids": torch.tensor([full_rej], dtype=torch.long, device=dev),
        "rej_mask": torch.ones((1, len(full_rej)), dtype=torch.long, device=dev),
        "prompt_len": len(p_ids),
    })

# 2. Step 8A: Precompute SFT Reference Log-Probs in ~60s with ZERO Extra Adapter in VRAM!
model.eval()
ref_chosen_list = []
ref_rejected_list = []

print("[*] Precomputing SFT reference log-probs against active SFT model (ZERO extra base/adapter VRAM)...")
with torch.inference_mode(), torch.amp.autocast("cuda", dtype=torch.float16 if torch.cuda.is_available() else torch.float32):
    for pair in tqdm(dpo_tokenized, desc="Precomputing Ref Logps"):
        c_sum, _ = compute_completion_logps(model, pair["c_ids"], pair["c_mask"], pair["prompt_len"])
        rej_sum, _ = compute_completion_logps(model, pair["rej_ids"], pair["rej_mask"], pair["prompt_len"])
        ref_chosen_list.append(c_sum.item())
        ref_rejected_list.append(rej_sum.item())

print(f"[✓] Successfully precomputed {len(ref_chosen_list)} reference log-probs in CPU memory. Zero VRAM overhead!")

# 3. Step 8B: Native DPO Training Loop with RPO Anchor
beta = 0.1
rpo_alpha = 0.2
grad_accum = 8
total_steps = len(dpo_tokenized) // grad_accum
learning_rate = 5e-6
warmup_steps = 10

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=learning_rate, weight_decay=0.01)

def get_dpo_lr(step):
    if step < warmup_steps:
        return learning_rate * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return learning_rate * 0.5 * (1.0 + math.cos(math.pi * progress))

scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

model.train()
if hasattr(model, 'config'):
    model.config.use_cache = False

print("[*] Starting Completion-Sliced QLoRA DPO training (Peak VRAM < 6.8 GB, 18 MB logits)...")
pbar = tqdm(total=total_steps, desc="DPO Progress")
running_loss = 0.0
optimizer.zero_grad()

for i, pair in enumerate(dpo_tokenized):
    with torch.amp.autocast("cuda", dtype=torch.float16 if torch.cuda.is_available() else torch.float32):
        pi_c_sum, pi_c_mean = compute_completion_logps(model, pair["c_ids"], pair["c_mask"], pair["prompt_len"])
        pi_rej_sum, _ = compute_completion_logps(model, pair["rej_ids"], pair["rej_mask"], pair["prompt_len"])

        ref_c = ref_chosen_list[i]
        ref_rej = ref_rejected_list[i]

        logits_diff = (pi_c_sum - ref_c) - (pi_rej_sum - ref_rej)
        dpo_loss = -torch.nn.functional.logsigmoid(beta * logits_diff)
        rpo_nll_loss = -pi_c_mean
        loss = (dpo_loss + rpo_alpha * rpo_nll_loss) / grad_accum

    scaler.scale(loss).backward()
    running_loss += loss.item() * grad_accum

    if (i + 1) % grad_accum == 0 or (i + 1) == len(dpo_tokenized):
        current_step = (i + 1) // grad_accum
        current_lr = get_dpo_lr(current_step)
        for param_group in optimizer.param_groups:
            param_group["lr"] = current_lr

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        avg_loss = running_loss / grad_accum
        pbar.set_postfix({"loss": f"{avg_loss:.4f}", "lr": f"{current_lr:.2e}"})
        pbar.update(1)
        running_loss = 0.0

pbar.close()

model.save_pretrained("checkpoints/vericall-dpo-lora")
tokenizer.save_pretrained("checkpoints/vericall-dpo-lora")
model.save_pretrained("checkpoints/dpo_adapter")
tokenizer.save_pretrained("checkpoints/dpo_adapter")
print("[✓] DPO training complete. Adapters saved to checkpoints/vericall-dpo-lora and checkpoints/dpo_adapter.")
del dpo_tokenized
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


### Stage 9: Final Evaluation & Statistical Rigor (Bootstrap CI & McNemar Tests)


In [ ]:
# Stage 9: Final Evaluation & Statistical Rigor (Bootstrap CI & McNemar Paired Tests)
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
if hasattr(model, 'config'):
    model.config.use_cache = True
if hasattr(model, 'eval'):
    model.eval()

from vericall.eval.stats import compute_mcnemar_test

device = next(model.parameters()).device

tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# --- 25-SAMPLE PRE-FLIGHT GATE ---
print("\n[*] Running 25-sample DPO Pre-Flight Smoke Check (batched)...")
smoke_dpo_prompts = [item["prompt"] for item in test_records[:25]]
smoke_dpo_preds = generate_batched(model, tokenizer, smoke_dpo_prompts, batch_size=8, max_new_tokens=140, desc="DPO Smoke (25)")

smoke_dpo_res = evaluate_predictions(test_records[:25], smoke_dpo_preds, "dpo_smoke")
dpo_smoke_acc = smoke_dpo_res["accuracy_pct"]
print(f"[DPO Smoke Check] Passed: {smoke_dpo_res['passed_requests']}/25 ({dpo_smoke_acc}%) vs SFT ({sft_smoke_acc}%)")
print(f"                  Errors: {smoke_dpo_res['error_taxonomy']}")

if dpo_smoke_acc < sft_smoke_acc:
    print(f"[!] Note: DPO smoke check on 25 samples is {dpo_smoke_acc}% (SFT: {sft_smoke_acc}%). Proceeding to full M=500 evaluation.")
else:
    print(f"[✓] PRE-FLIGHT GATE PASSED: DPO ({dpo_smoke_acc}%) matches/exceeds SFT ({sft_smoke_acc}%).")

# --- FULL 500-SAMPLE EVALUATION (BATCHED) ---
print("\n[*] Generating final predictions with VeriCall (SFT+DPO) on full M=500 unseen APIs (batched, 8x faster)...")
rest_dpo_prompts = [item["prompt"] for item in test_records[25:]]
rest_dpo_preds = generate_batched(model, tokenizer, rest_dpo_prompts, batch_size=8, max_new_tokens=140, desc="DPO Full Eval (475)")
dpo_predictions = smoke_dpo_preds + rest_dpo_preds

dpo_results = evaluate_predictions(test_records, dpo_predictions, "sft_dpo_vericall")

# --- BFCL AST SUBSET EVALUATION (VERICALL / DPO) ---
print("\n[*] Evaluating VeriCall (SFT+DPO) on BFCL AST subset (N=200, batched)...")
bfcl_dpo_preds = generate_batched(model, tokenizer, bfcl_prompts, batch_size=8, max_new_tokens=140, desc="DPO BFCL Eval (200)")
bfcl_dpo_results = evaluate_predictions(bfcl_records, bfcl_dpo_preds, "bfcl_dpo")
bfcl_dpo_acc_pct = bfcl_dpo_results["accuracy_pct"]
print(f"[VeriCall BFCL]  Accuracy: {bfcl_dpo_acc_pct}% (Passed: {bfcl_dpo_results['passed_requests']}/{bfcl_dpo_results['total_requests']})")

if dpo_results["accuracy_pct"] < sft_results["accuracy_pct"]:
    print(f"\n[!] WARNING: DPO accuracy ({dpo_results['accuracy_pct']}%) is lower than SFT accuracy ({sft_results['accuracy_pct']}%).")
    print(f"    Possible post-training regression detected. SFT checkpoint is preserved in checkpoints/sft_adapter.")
else:
    print(f"\n[✓] POST-TRAINING ALIGNMENT SUCCESS: DPO accuracy ({dpo_results['accuracy_pct']}%) >= SFT ({sft_results['accuracy_pct']}%).")

# Compute paired McNemar tests
stat_base, p_base = compute_mcnemar_test(base_results["binary_scores"], dpo_results["binary_scores"])
stat_sft, p_sft = compute_mcnemar_test(sft_results["binary_scores"], dpo_results["binary_scores"])

print(f"\n=======================================================")
print(f"FINAL VERICALL BENCHMARK RESULTS (M=500 Unseen APIs)")
print(f"=======================================================")
print(f"[Base Model]     Accuracy: {base_results['accuracy_pct']}%  95% CI: [{base_results['bootstrap_95_ci'][0]}%, {base_results['bootstrap_95_ci'][1]}%]")
print(f"[Base + Constr]  Accuracy: {constrained_results['accuracy_pct']}%  95% CI: [{constrained_results['bootstrap_95_ci'][0]}%, {constrained_results['bootstrap_95_ci'][1]}%]")
print(f"[SFT-Only]       Accuracy: {sft_results['accuracy_pct']}%  95% CI: [{sft_results['bootstrap_95_ci'][0]}%, {sft_results['bootstrap_95_ci'][1]}%]")
print(f"[VeriCall (DPO)] Accuracy: {dpo_results['accuracy_pct']}%  95% CI: [{dpo_results['bootstrap_95_ci'][0]}%, {dpo_results['bootstrap_95_ci'][1]}%]")
print(f"\nMcNemar Test vs Base:    chi2 = {stat_base:.4f}, p = {p_base:.4e} (Significant: {p_base < 0.05})")
print(f"McNemar Test vs SFT:     chi2 = {stat_sft:.4f}, p = {p_sft:.4e} (Significant: {p_sft < 0.05})")
print(f"-------------------------------------------------------")
print(f"BFCL AST Subset Results (N=200):")
print(f"  Base Model:       {bfcl_base_acc_pct}%")
print(f"  SFT-Only:         {bfcl_sft_acc_pct}%")
print(f"  VeriCall (DPO):   {bfcl_dpo_acc_pct}%")
print(f"=======================================================")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


### Stage 10: Save Evaluation Summary & Render Results


In [ ]:
from pathlib import Path
import json
import os
import shutil

summary_data = {
    "metadata": {
        "model_base": model_id,
        "eval_set": "eval_unseen_apis.jsonl",
        "eval_count": len(test_records),
        "bfcl_set": "eval_bfcl_ast_subset.jsonl",
        "bfcl_count": len(bfcl_records),
        "mode": "full_benchmark_v2",
        "device": str(device),
        "num_dpo_pairs": len(dpo_pairs),
    },
    "configs": {
        "base_model": base_results,
        "base_constrained": constrained_results,
        "sft_only": sft_results,
        "sft_dpo_vericall": dpo_results,
    },
    "bfcl_results": {
        "base_model": bfcl_base_results,
        "sft_only": bfcl_sft_results,
        "sft_dpo_vericall": bfcl_dpo_results,
    },
    "bfcl_base_acc_pct": bfcl_base_acc_pct,
    "bfcl_sft_acc_pct": bfcl_sft_acc_pct,
    "bfcl_dpo_acc_pct": bfcl_dpo_acc_pct,
    "mcnemar_tests": {
        "vericall_vs_base": {"statistic": stat_base, "p_value": p_base},
        "vericall_vs_sft": {"statistic": stat_sft, "p_value": p_sft},
    },
    "mcnemar_vs_base": {"statistic": stat_base, "p_value": p_base},
    "mcnemar_vs_sft": {"statistic": stat_sft, "p_value": p_sft}
}

Path("results").mkdir(exist_ok=True)
with open("results/eval_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=2)

print("[✓] Saved evaluation summary to results/eval_summary.json")

# Copy to /kaggle/working for direct download
if os.path.exists("/kaggle/working"):
    shutil.copy2("results/eval_summary.json", "/kaggle/working/eval_summary.json")
    print("[✓] Copied to /kaggle/working/eval_summary.json for direct artifact download")

!python scripts/render_results.py
